# 05 — Detailed National Analysis

Deep-dive into a single country. Includes:
1. Spatial maps of productivity loss, population, and wealth
2. Concentration curve with CI annotation
3. Productivity loss by wealth quintile
4. Year-by-year time series of mean productivity loss
5. Sub-national CI (admin level 1)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import geopandas as gpd
import rasterio
import rasterio.mask
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from scripts.inequality import calculate_CI, calculate_concentration_curve, calculate_quantile_ratio, prepare_arrays
from scripts.raster_utils import get_country_geometry, align_rasters

with open(ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

DATA       = ROOT / 'data'
ANNUAL_DIR = DATA / 'processed' / 'annual'
RESULTS    = ROOT / 'results'

print('Ready.')

## Configuration

In [ ]:
# --- Select country and epoch ---
ISO3        = 'KEN'
EPOCH_START = config['wbgt']['start_year']
EPOCH_END   = config['wbgt']['end_year']
# --------------------------------

POP_PATH        = str(DATA / config['data']['population'])
RWI_PATH        = str(DATA / config['data']['rwi'])
BOUNDARIES_PATH = str(DATA / config['data']['boundaries'])
EPOCH_RISK_PATH = str(DATA / 'processed' / f'productivity_loss_{EPOCH_START}_{EPOCH_END}.tif')

print(f'Country: {ISO3}  |  Epoch: {EPOCH_START}–{EPOCH_END}')

## Load and align country data

In [ ]:
geom = get_country_geometry(BOUNDARIES_PATH, ISO3)
pop, rwi, risk = align_rasters(POP_PATH, RWI_PATH, EPOCH_RISK_PATH, geom)

arrays = prepare_arrays(pop, rwi, risk)
if arrays is None:
    raise ValueError('Insufficient valid data for this country.')
pop_f, rwi_f, risk_f = arrays

ci = calculate_CI(pop_f, rwi_f, risk_f)
qr = calculate_quantile_ratio(pop_f, rwi_f, risk_f)
mean_loss = float(np.average(risk_f, weights=pop_f))

print(f'Concentration Index:      {ci:.3f}')
print(f'Quantile Ratio (Q5/Q1):   {qr:.3f}')
print(f'Mean productivity loss:   {mean_loss:.4f}')
print(f'Population (with RWI):    {pop_f.sum():,.0f}')

## Figure 1 — Spatial maps

In [ ]:
# Load country boundary for overlay
world = gpd.read_file(BOUNDARIES_PATH)
for col in ['ADM0_A3', 'ISO_A3', 'iso_a3', 'GID_0']:
    if col in world.columns:
        iso_col = col
        break
country_gdf = world[world[iso_col] == ISO3]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Mask zero-population cells for cleaner maps
pop_masked  = np.where(pop > 0, pop, np.nan)
risk_masked = np.where(pop > 0, risk, np.nan)
rwi_masked  = np.where((pop > 0) & ~np.isnan(rwi), rwi, np.nan)

panels = [
    (axes[0], np.log1p(pop_masked),  'viridis',  None,  None,  'Population (log scale)'),
    (axes[1], risk_masked,            'YlOrRd',   0,     None,  'Mean annual productivity loss'),
    (axes[2], rwi_masked,             'RdYlGn',   -2,    2,     'Relative Wealth Index'),
]

for ax, arr, cmap, vmin, vmax, title in panels:
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title)
    ax.axis('off')

fig.suptitle(f'{ISO3} — {EPOCH_START}–{EPOCH_END}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / f'{ISO3}_maps_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — Concentration curve

In [ ]:
cum_pop, cum_risk = calculate_concentration_curve(pop_f, rwi_f, risk_f)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(cum_pop, cum_risk, color='steelblue', linewidth=2, label=f'{ISO3} (CI = {ci:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Line of equality')
ax.fill_between(cum_pop, cum_risk, cum_pop,
                alpha=0.15, color='steelblue' if ci < 0 else 'tomato')

ax.set_xlabel('Cumulative population share (poorest → wealthiest)', fontsize=11)
ax.set_ylabel('Cumulative productivity loss share', fontsize=11)
ax.set_title(f'Concentration Curve — {ISO3} ({EPOCH_START}–{EPOCH_END})', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Annotate CI and QR
ax.text(0.05, 0.92, f'CI = {ci:.3f}\nQR (Q5/Q1) = {qr:.2f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(RESULTS / f'{ISO3}_concentration_curve_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — Productivity loss by wealth quintile

In [ ]:
df = pd.DataFrame({'pop': pop_f, 'rwi': rwi_f, 'risk': risk_f})
df = df.sort_values('rwi').reset_index(drop=True)
df['cum_pop'] = df['pop'].cumsum()
total_pop = df['pop'].sum()

quintile_results = []
for q in range(1, 6):
    mask = (df['cum_pop'] > (q - 1) / 5 * total_pop) & (df['cum_pop'] <= q / 5 * total_pop)
    sub = df[mask]
    if sub.empty:
        continue
    weighted_loss = np.average(sub['risk'], weights=sub['pop'])
    quintile_results.append({'quintile': f'Q{q}', 'mean_loss': weighted_loss, 'pop': sub['pop'].sum()})

qdf = pd.DataFrame(quintile_results)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(qdf['quintile'], qdf['mean_loss'],
              color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(qdf))),
              edgecolor='white', linewidth=0.5)
ax.axhline(mean_loss, color='black', linewidth=1.2, linestyle='--', label=f'National mean ({mean_loss:.4f})')
ax.set_xlabel('Wealth quintile (Q1 = poorest, Q5 = wealthiest)')
ax.set_ylabel('Mean annual productivity loss')
ax.set_title(f'Productivity Loss by Wealth Quintile — {ISO3} ({EPOCH_START}–{EPOCH_END})')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / f'{ISO3}_quintile_loss_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
plt.show()

print(qdf.to_string(index=False))

## Figure 4 — Year-by-year time series

In [ ]:
annual_files = sorted(ANNUAL_DIR.glob('productivity_loss_*.tif'))
years = [int(f.stem.split('_')[-1]) for f in annual_files]

ts_results = []
for year, fpath in zip(years, annual_files):
    from scripts.raster_utils import clip_raster_to_geometry, resample_to_reference
    from rasterio.warp import Resampling

    # Clip annual risk raster to country and align to pop grid
    risk_raw, risk_meta = clip_raster_to_geometry(str(fpath), geom)

    # We already have pop aligned — use same pop_meta from align_rasters
    # Re-run align for this year's risk only
    _, _, risk_yr = align_rasters(POP_PATH, RWI_PATH, str(fpath), geom)

    arrays_yr = prepare_arrays(pop, rwi, risk_yr)
    if arrays_yr is None:
        continue
    pop_yr, rwi_yr, risk_yr_f = arrays_yr

    ts_results.append({
        'year': year,
        'mean_loss': float(np.average(risk_yr_f, weights=pop_yr)),
        'CI': calculate_CI(pop_yr, rwi_yr, risk_yr_f),
    })

ts_df = pd.DataFrame(ts_results).sort_values('year')

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax = axes[0]
ax.plot(ts_df['year'], ts_df['mean_loss'], marker='o', markersize=4, color='tomato')
ax.set_ylabel('Mean annual productivity loss')
ax.set_title(f'{ISO3} — Annual productivity loss and inequality')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(ts_df['year'], ts_df['CI'], marker='o', markersize=4, color='steelblue')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Concentration Index')
ax.set_xlabel('Year')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / f'{ISO3}_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 5 — Sub-national CI (Admin level 1)

Requires admin level 1 boundaries. Adjust `ADM1_PATH` to your file.

In [ ]:
# Path to admin level 1 boundaries (e.g. geoBoundaries or GADM)
# Expected to contain an ISO3 column and geometry
ADM1_PATH = DATA / 'boundaries' / f'{ISO3}_adm1.gpkg'  # adjust as needed

if not ADM1_PATH.exists():
    print(f'Admin-1 boundary file not found: {ADM1_PATH}')
    print('Skipping sub-national analysis.')
else:
    adm1 = gpd.read_file(ADM1_PATH)
    print(f'Admin-1 regions: {len(adm1)}')
    print('Columns:', list(adm1.columns))

    # Identify name column
    name_col = next((c for c in ['shapeName', 'NAME_1', 'name', 'ADM1_EN'] if c in adm1.columns), adm1.columns[0])

    adm1_results = []
    for _, region in adm1.iterrows():
        try:
            pop_r, rwi_r, risk_r = align_rasters(POP_PATH, RWI_PATH, EPOCH_RISK_PATH, region.geometry)
            arrays_r = prepare_arrays(pop_r, rwi_r, risk_r)
            if arrays_r is None:
                continue
            p, w, r = arrays_r
            adm1_results.append({
                'name': region[name_col],
                'CI': calculate_CI(p, w, r),
                'mean_loss': float(np.average(r, weights=p)),
                'population': float(p.sum()),
                'geometry': region.geometry,
            })
        except Exception as e:
            print(f'  {region[name_col]}: {e}')

    adm1_gdf = gpd.GeoDataFrame(adm1_results, geometry='geometry', crs=adm1.crs)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    adm1_gdf.plot(ax=axes[0], column='mean_loss', cmap='YlOrRd',
                  edgecolor='white', linewidth=0.5, legend=True,
                  legend_kwds={'label': 'Mean productivity loss', 'shrink': 0.7})
    axes[0].set_title('Mean Annual Productivity Loss')
    axes[0].axis('off')

    adm1_gdf.plot(ax=axes[1], column='CI', cmap='RdBu', vmin=-1, vmax=1,
                  edgecolor='white', linewidth=0.5, legend=True,
                  legend_kwds={'label': 'Concentration Index', 'shrink': 0.7})
    axes[1].set_title('Concentration Index')
    axes[1].axis('off')

    fig.suptitle(f'{ISO3} — Sub-national Analysis ({EPOCH_START}–{EPOCH_END})', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS / f'{ISO3}_subnational_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(adm1_gdf[['name', 'CI', 'mean_loss', 'population']].sort_values('CI').to_string(index=False))